[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ababber/pyhou-02-17-2026/blob/main/part-2-deep-learning/research-notebook.ipynb)

> **What You'll Need**
> - **To run demos locally:** Python 3.9+, tensorflow, numpy, sklearn ([see requirements](../requirements.txt))
> - **To run demos in Colab:** Nothing — click the badge above, TensorFlow is pre-installed
> - **To run the trading strategy:** Free [QuantConnect](https://www.quantconnect.com/) account (no credit card required)
> - **Time:** ~45 min to read, ~10 min to run backtest on QC (model training is slow)



# Part 2: Deep Learning — Temporal CNN

**From Linear Models to Learned Features**

In Part 1, ridge regression failed to generate alpha — it couldn't find predictive signal in hand-crafted volatility features. This notebook asks: what if we let the model *learn* the features?

We'll build a convolutional neural network (CNN) that takes raw OHLCV data and learns to predict price direction. Spoiler: it works. But the *why* is more interesting than the *what*.

**What you'll learn:**
- Why CNNs work on sequential data (not just images)
- How temporal splitting lets the model weight recent vs. distant patterns differently
- How to interpret confidence-weighted position sizing
- Why deep learning succeeds where linear models fail (and where it still struggles)

**Prerequisites:**
- Part 1 of this series (for backtest metric interpretation)
- Basic Python (numpy, pandas)
- Familiarity with neural network concepts (layers, activation, training)

---

**Navigation:** [The Strategy](#2-the-strategy) | [The Math](#3-the-math) | [Implementation](#4-implementation) | [Results](#5-results) | [Analysis](#6-analysis) | [References](#7-references)

## 1. Introduction

### The Question

Can a neural network find patterns in price data that linear models miss?

Part 1 established our baseline: ridge regression on volatility features produced a Sharpe of 0.212 with *negative* alpha (-0.062). The model was just riding market beta — no independent signal.

| Part | Model | Alpha | Beta | Verdict |
|------|-------|-------|------|--------|
| 1 | Ridge Regression | -0.062 | 1.146 | Beta masquerade |
| 2 | Temporal CNN | 0.093 | 0.278 | **Genuine alpha** |
| 3 | Chronos (coming) | ? | ? | Foundation model |

Same platform. Same time period. But very different results.

### Why a CNN for Time Series?

CNNs are famous for image recognition. Why use them on price data?

**The key insight:** A CNN's convolutional layer is really just a *pattern detector*. It slides a small window across the input, looking for specific shapes. In an image, those shapes might be edges or curves. In a price series, they might be:

- Gap-and-fade patterns (price gaps then reverses)
- Volume spikes preceding moves
- OHLC relationships (long wicks, dojis, engulfing patterns)

The CNN doesn't know these are "patterns" — it just learns that certain input configurations predict certain outputs.

> **Key Insight:** A 1D CNN treats a price series like a narrow image — one pixel wide, T pixels tall, with 5 color channels (OHLCV). The convolution slides vertically through time, detecting patterns across all channels simultaneously.

**Why not an RNN/LSTM?** LSTMs are designed for sequences where order matters and long-range dependencies exist. For short-term price prediction (5-day lookahead), local patterns may matter more than long memory. CNNs are also faster to train and less prone to vanishing gradients.

## 2. The Strategy

### Universe: Top 3 QQQ Components

Unlike Part 1's diversified futures basket, this strategy focuses on mega-cap tech:

| Selection | Method |
|-----------|--------|
| Universe | QQQ ETF constituents |
| Filter | Top 3 by ETF weight |
| Update | Weekly (Monday) |

**Why so narrow?** The CNN learns stock-specific patterns. Training on 3 stocks (rather than 100) means each model sees enough data to learn, and we can retrain weekly without excessive compute.

**Typical holdings:** Apple, Microsoft, NVIDIA (or whoever currently dominates QQQ by weight).

### Features: Raw OHLCV

Part 1 used engineered features (volatility, ATR, open interest). Here, we let the CNN figure it out:

| Feature | Raw Input | What the CNN might learn |
|---------|-----------|-------------------------|
| Open | Daily open price | Gap direction |
| High | Daily high | Intraday strength |
| Low | Daily low | Intraday weakness |
| Close | Daily close | Net direction |
| Volume | Shares traded | Conviction behind moves |

**Input shape:** 15 days × 5 features = a (15, 5) matrix per prediction.

**Critical preprocessing:** Raw OHLCV values span wildly different scales (price ~$100-500, volume ~millions). We apply `StandardScaler` to normalize each feature to mean=0, std=1. Without this, the CNN's gradients would be dominated by the largest-scale features.

### Labels: Three-Class Direction

We're predicting *direction*, not *magnitude*:

| Label | Condition | Meaning |
|-------|-----------|----------|
| UP | 5-day avg close > current close by >0.01% | Price rising |
| DOWN | 5-day avg close < current close by >0.01% | Price falling |
| STATIONARY | Change within ±0.01% | No clear direction |

**Why 5-day average?** Smooths out daily noise. We're not predicting tomorrow's close — we're predicting the trend over the next trading week.

**Why three classes?** The STATIONARY class is crucial. It lets the model express uncertainty. When the model can't decide between UP and DOWN, it predicts STATIONARY, and *we don't trade*. This is built-in risk management.

### Allocation: Confidence-Weighted

Part 1 weighted positions by *inverse predicted volatility*. Part 2 weights by *model confidence*:

```
weight_i = direction_i × confidence_i
```

Where:
- `direction_i` = +1 for UP, -1 for DOWN
- `confidence_i` = softmax probability of the predicted class

**Threshold:** Only trade if confidence > 55%. With 3 classes, random chance is 33%. We require the model to be substantially more confident than chance.

**Normalization:** If total absolute weight exceeds 1, we scale down proportionally. The portfolio is never more than 100% invested.

**Schedule:**
- **Train:** Every Monday at 9:00 AM (20 epochs on trailing 500 days)
- **Predict:** Every Monday, 2 minutes after market open
- **Rebalance:** Full liquidation + re-entry each week

## 3. The Math

Three concepts drive this strategy: **convolution** (pattern detection), **temporal splitting** (time-aware features), and **softmax classification** (probabilistic predictions).

### 1D Convolution: The Pattern Detector

A 1D convolution slides a *kernel* (small weight matrix) across the input sequence, computing a weighted sum at each position.

For input $x$ of length $T$ with $F$ features, and a kernel of size $k$:

$$y_t = \text{ReLU}\left(\sum_{i=0}^{k-1} \sum_{f=0}^{F-1} W_{i,f} \cdot x_{t+i,f} + b\right)$$

| Term | Meaning |
|------|---------|
| $W_{i,f}$ | Kernel weight for position $i$, feature $f$ |
| $x_{t+i,f}$ | Input value at time $t+i$, feature $f$ |
| $b$ | Bias term |
| ReLU | $\max(0, z)$ — zeroes negative values |

**Output shape:** With input (15, 5) and kernel size 4, we get (15-4+1, N) = (12, N) where N is the number of filters.

### Demo: How Conv1D Detects Patterns

Run this cell to see how a 1D convolution responds to different input patterns.

In [ ]:
# RUNNABLE DEMO: Conv1D pattern detection
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, Input
from tensorflow.keras import Model

np.random.seed(42)
tf.random.set_seed(42)

# Create a simple Conv1D with 1 filter, kernel size 3
inputs = Input(shape=(10, 1))  # 10 timesteps, 1 feature
conv = Conv1D(1, 3, activation='relu', use_bias=False)(inputs)
model = Model(inputs, conv)

# Manually set kernel to detect "up-down-up" pattern
kernel = np.array([[[1.0], [-2.0], [1.0]]])  # Shape: (kernel_size, input_features, filters)
model.layers[1].set_weights([kernel.transpose(0, 2, 1).reshape(3, 1, 1)])

# Test on different patterns
patterns = {
    'Flat':      np.array([5, 5, 5, 5, 5, 5, 5, 5, 5, 5]),
    'Uptrend':   np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]),
    'V-shape':   np.array([5, 4, 3, 2, 1, 2, 3, 4, 5, 6]),  # Contains up-down-up!
    'Spike':     np.array([5, 5, 5, 5, 9, 5, 5, 5, 5, 5]),
}

print("Kernel: [+1, -2, +1] — detects V-shaped reversals")
print("="*50)

for name, seq in patterns.items():
    x = seq.reshape(1, 10, 1).astype(np.float32)
    output = model.predict(x, verbose=0)
    max_activation = output.max()
    print(f"{name:12} → Max activation: {max_activation:.2f}")

print("\n→ The V-shape has highest activation (the kernel 'fires' on reversals)")

### Temporal Splitting: Time-Aware Features

This is the key architectural innovation. After the first Conv1D extracts 30 feature maps of length 12, we **split them into three temporal regions**:

| Region | Positions | Represents |
|--------|-----------|------------|
| Long-term | 0–3 | Oldest patterns (11-15 days ago) |
| Mid-term | 4–7 | Middle patterns (6-10 days ago) |
| Short-term | 8–11 | Recent patterns (1-5 days ago) |

Each region then passes through its own 1×1 convolution:

```python
long_term_conv = Conv1D(1, 1, activation='relu')(long_term)
mid_term_conv = Conv1D(1, 1, activation='relu')(mid_term)
short_term_conv = Conv1D(1, 1, activation='relu')(short_term)
```

**Why does this matter?** A 1×1 convolution with 30 input channels and 1 output channel is essentially a *learned weighted average* of those 30 features. By having separate 1×1 convs for each temporal region, the model can learn:

- "For recent data, volume matters most"
- "For older data, the high-low spread matters most"
- "Mid-term momentum is the key signal"

The three outputs are concatenated and fed to the classification head.

In [ ]:
# RUNNABLE DEMO: Temporal splitting visualization
import numpy as np
import tensorflow as tf

# Simulate the temporal split on feature extraction output
# Pretend we have 12 time positions with 30 features each
np.random.seed(42)
feature_maps = np.random.randn(1, 12, 30)  # (batch, time, features)

# Split into 3 regions of 4 time steps each
splits = tf.split(feature_maps, num_or_size_splits=3, axis=1)
long_term, mid_term, short_term = splits

print("Feature extraction output shape:", feature_maps.shape)
print("After temporal split:")
print(f"  Long-term (oldest):  {long_term.shape} — positions 0-3")
print(f"  Mid-term:            {mid_term.shape} — positions 4-7")
print(f"  Short-term (recent): {short_term.shape} — positions 8-11")

print("\n→ Each region gets its own 1×1 conv to learn time-specific feature weights")

### Softmax Classification

The final Dense layer has 3 outputs (UP, DOWN, STATIONARY) with softmax activation:

$$P(\text{class}_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Softmax converts raw scores into probabilities that sum to 1. The predicted class is the one with highest probability; that probability is our **confidence**.

| Raw scores | Softmax | Interpretation |
|------------|---------|----------------|
| [2.0, 0.5, 0.3] | [0.71, 0.16, 0.13] | 71% confident UP |
| [1.0, 1.0, 1.0] | [0.33, 0.33, 0.33] | No confidence — STATIONARY |

**Loss function:** Categorical cross-entropy measures how far the predicted distribution is from the true label (one-hot encoded):

$$L = -\sum_i y_i \log(\hat{y}_i)$$

When the true label is UP (y = [1,0,0]), the loss is just $-\log(P(\text{UP}))$. The model is penalized for low confidence in the correct class.

In [ ]:
# RUNNABLE DEMO: Softmax and confidence
import numpy as np

def softmax(z):
    exp_z = np.exp(z - np.max(z))  # Subtract max for numerical stability
    return exp_z / exp_z.sum()

# Test cases: raw model outputs (logits)
scenarios = {
    'Strong UP signal':    [3.0, 0.5, 0.2],
    'Weak UP signal':      [1.2, 0.9, 0.8],
    'Uncertain':           [1.0, 1.0, 1.0],
    'Strong DOWN signal':  [0.1, 2.5, 0.3],
}

print("Raw Scores → Softmax Probabilities → Trading Decision")
print("="*65)

labels = ['UP', 'DOWN', 'STAT']
threshold = 0.55

for name, logits in scenarios.items():
    probs = softmax(np.array(logits))
    pred_idx = np.argmax(probs)
    confidence = probs[pred_idx]
    
    # Trading decision
    if labels[pred_idx] == 'STAT' or confidence < threshold:
        decision = "NO TRADE"
    else:
        decision = f"{labels[pred_idx]} (weight={confidence:.0%})"
    
    print(f"{name:20} → [{probs[0]:.0%}, {probs[1]:.0%}, {probs[2]:.0%}] → {decision}")

print(f"\n→ Only trade when confidence > {threshold:.0%} and prediction ≠ STATIONARY")

### StandardScaler: Why Normalization Matters

Raw OHLCV data has wildly different scales:

| Feature | Typical Range | Scale |
|---------|--------------|-------|
| Open/High/Low/Close | $100 – $500 | ~10² |
| Volume | 10M – 100M | ~10⁷ |

Without normalization, volume would dominate the gradients. The model would effectively ignore price patterns.

**StandardScaler** transforms each feature to mean=0, std=1:

$$z = \frac{x - \mu}{\sigma}$$

**Important detail:** The scaler is fit on the training data (500 days) and applied to both training and prediction inputs. This prevents data leakage — the model never sees future statistics during training.

In [ ]:
# RUNNABLE DEMO: StandardScaler effect on OHLCV-like data
import numpy as np
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Simulate 15 days of OHLCV data (typical ranges)
n_days = 15
data = np.column_stack([
    np.random.uniform(150, 160, n_days),      # Open
    np.random.uniform(155, 165, n_days),      # High
    np.random.uniform(145, 155, n_days),      # Low
    np.random.uniform(150, 160, n_days),      # Close
    np.random.uniform(20e6, 50e6, n_days),    # Volume (millions)
])

print("Before StandardScaler:")
print(f"  Open  range: {data[:,0].min():.1f} – {data[:,0].max():.1f}")
print(f"  Volume range: {data[:,4].min()/1e6:.1f}M – {data[:,4].max()/1e6:.1f}M")
print(f"  Volume/Open ratio: {data[:,4].mean() / data[:,0].mean():.0f}x")

# Apply StandardScaler
scaler = StandardScaler()
scaled = scaler.fit_transform(data)

print("\nAfter StandardScaler:")
print(f"  All features: mean ≈ 0, std ≈ 1")
print(f"  Open  range: {scaled[:,0].min():.2f} – {scaled[:,0].max():.2f}")
print(f"  Volume range: {scaled[:,4].min():.2f} – {scaled[:,4].max():.2f}")

print("\n→ Now all features contribute equally to gradients during training")

## 4. Implementation

This section walks through the QuantConnect algorithm. The code runs on QC's platform — you can't execute it locally, but you can copy it to QC to run backtests.

> **Note:** Code cells marked `# QUANTCONNECT` are view-only. Copy to [QuantConnect](https://www.quantconnect.com/) to execute.
>
> **QC Free Tier:** This strategy uses more compute than Part 1 (weekly CNN training), but still runs on the free tier. Training may take several minutes per backtest.

### Step 1: Algorithm Setup

Configure the universe (QQQ top 3), training schedule, and data handling.

In [ ]:
# QUANTCONNECT — Algorithm Setup (view-only)
# This code runs on QuantConnect, not in Colab. Copy to QC to execute.

try:
    from AlgorithmImports import *
    from temporalcnn import TemporalCNN, Direction, factor_names

    class TemporalCNNPredictionAlgorithm(QCAlgorithm):

        def initialize(self):
            # Backtest period (same as Part 1 for comparison)
            self.set_start_date(2018, 12, 31)
            self.set_end_date(2024, 4, 1)
            self.set_cash(100_000)  # $100K starting capital

            # Model parameters
            self._training_samples = self.get_parameter("training_samples", 500)
            self._universe_size = self.get_parameter("universe_size", 3)

            # Universe: QQQ ETF components, filtered by weight
            etf = Symbol.create("QQQ", SecurityType.EQUITY, Market.USA)
            date_rule = self.date_rules.week_start(etf)
            self.universe_settings.schedule.on(date_rule)
            self.universe_settings.data_normalization_mode = DataNormalizationMode.RAW
            self.universe_settings.asynchronous = True
            self._universe = self.add_universe(
                self.universe.etf(etf, universe_filter_func=self._select_assets)
            )

            # Schedule: train Monday 9am, trade Monday after open
            self.train(date_rule, self.time_rules.at(9, 0), self._update_models)
            self.schedule.on(
                date_rule, self.time_rules.after_market_open(etf, 2), self._trade
            )

except ModuleNotFoundError:
    print("ℹ️ This cell contains QuantConnect code (view-only in Colab).")
    print("   Copy to QuantConnect → Algorithm Lab → New Algorithm to run.")

### Step 2: Universe Selection

Select the top 3 QQQ components by ETF weight. These are typically Apple, Microsoft, and NVIDIA.

In [ ]:
# QUANTCONNECT — Universe Selection (view-only)

def _select_assets(self, constituents):
    """Select top N assets by ETF weight."""
    # Filter out constituents without weight data
    constituents = [c for c in constituents if c.weight]
    
    if constituents:
        # Sort by weight, take top N
        return [
            c.symbol
            for c in sorted(constituents, key=lambda c: c.weight)[-self._universe_size:]
        ]
    
    return Universe.UNCHANGED

### Step 3: The CNN Model

The model architecture lives in `temporalcnn.py`. Here's the key method that builds the network:

In [ ]:
# QUANTCONNECT — Model Architecture (view-only)

def _create_model(self):
    """Creates the temporal CNN architecture."""
    inputs = Input(shape=(self._n_tsteps, len(factor_names)))  # (15, 5)

    # Feature extraction: Conv1D with 30 filters, kernel size 4
    # Output shape: (12, 30)
    feature_extraction = Conv1D(30, 4, activation='relu')(inputs)

    # TEMPORAL SPLIT: Divide into 3 equal time regions
    long_term = Lambda(f0, output_shape=(4, 30))(feature_extraction)   # Oldest
    mid_term = Lambda(f1, output_shape=(4, 30))(feature_extraction)    # Middle
    short_term = Lambda(f2, output_shape=(4, 30))(feature_extraction)  # Recent

    # Each region gets its own 1×1 conv (learned feature weighting)
    long_term_conv = Conv1D(1, 1, activation='relu')(long_term)
    mid_term_conv = Conv1D(1, 1, activation='relu')(mid_term)
    short_term_conv = Conv1D(1, 1, activation='relu')(short_term)

    # Recombine: concatenate back to (12, 1)
    combined = Concatenate(axis=1)([long_term_conv, mid_term_conv, short_term_conv])

    # Classification head
    flattened = Flatten()(combined)  # (12,)
    outputs = Dense(3, activation='softmax')(flattened)  # [P(UP), P(DOWN), P(STAT)]

    return Model(inputs=inputs, outputs=outputs)

### Step 4: Data Preparation & Training

The model is retrained weekly on 500 days of history. Key steps: label construction, windowing, scaling.

In [ ]:
# QUANTCONNECT — Data Preparation (view-only)

def _prepare_data(self, data, rolling_avg_window_size=5, stationary_threshold=.0001):
    """Prepare OHLCV data for CNN training."""
    df = data[factor_names]  # ['open', 'high', 'low', 'close', 'volume']
    shift = -(rolling_avg_window_size - 1)

    # LABEL CONSTRUCTION
    # Compute 5-day rolling average of close, shifted backward
    df['close_avg'] = df['close'].rolling(window=rolling_avg_window_size).mean().shift(shift)
    df['close_avg_change_pct'] = (df['close_avg'] - df['close']) / df['close']

    def label_data(row):
        if row['close_avg_change_pct'] > stationary_threshold:
            return Direction.UP
        elif row['close_avg_change_pct'] < -stationary_threshold:
            return Direction.DOWN
        else:
            return Direction.STATIONARY

    df['movement_labels'] = df.apply(label_data, axis=1)

    # WINDOWING: Create 15-day sliding windows
    data_windows = []
    labels = []
    for i in range(len(df) - self._n_tsteps + 1 + shift):
        label = df['movement_labels'].iloc[i + self._n_tsteps - 1]
        data_windows.append(df[factor_names].iloc[i:i + self._n_tsteps].values)
        labels.append(label)

    data_array = np.array(data_windows)  # Shape: (samples, 15, 5)

    # SCALING: StandardScaler requires 2D input
    dim1, dim2, dim3 = data_array.shape
    data_2d = data_array.reshape(dim1 * dim2, dim3)
    data_scaled = self._scaler.fit_transform(data_2d)
    data_array = data_scaled.reshape(dim1, dim2, dim3)

    # ONE-HOT ENCODE labels
    return data_array, utils.to_categorical(labels, num_classes=3)

### Step 5: Trading Logic

Get predictions, apply confidence threshold, build confidence-weighted portfolio.

In [ ]:
# QUANTCONNECT — Trading Logic (view-only)

def _trade(self):
    """Called every Monday — get predictions and rebalance."""
    weight_by_symbol = {}

    for symbol in self._universe.selected:
        security = self.securities[symbol]
        
        # Get last 15 days of data
        symbol_df = security.history.tail(15)
        
        # Predict direction and confidence
        prediction, confidence = security.cnn.predict(symbol_df)

        # Only trade if:
        # 1. Not STATIONARY
        # 2. Confidence > 55%
        # 3. Confidence is valid (not NaN)
        if (prediction != Direction.STATIONARY and
            not math.isnan(confidence) and
            confidence > 0.55):
            
            # Direction factor: +1 for UP, -1 for DOWN
            factor = -1 if prediction == Direction.DOWN else 1
            weight_by_symbol[security.symbol] = factor * confidence

        self.plot("Confidence", str(security.symbol.id), confidence)

    # Normalize weights (absolute sum ≤ 1)
    weight_sum = sum(abs(w) for w in weight_by_symbol.values())
    weight_factor = 1 if weight_sum <= 1 else 1 / weight_sum

    # Execute: liquidate all, then enter new positions
    portfolio_targets = [
        PortfolioTarget(symbol, weight * weight_factor)
        for symbol, weight in weight_by_symbol.items()
    ]
    self.set_holdings(portfolio_targets, True)  # True = liquidate first

## 5. Results

### Backtest Summary

| Metric | Part 1 (Ridge) | Part 2 (CNN) | Improvement |
|--------|----------------|--------------|-------------|
| Net Profit | 34.8% | **145.2%** | +317% |
| CAGR | 5.85% | **18.6%** | +218% |
| Sharpe | 0.212 | **0.649** | +206% |
| Max Drawdown | 54.7% | **26.3%** | -52% (better) |
| Alpha | -0.062 | **0.093** | Negative → Positive |
| Beta | 1.146 | **0.278** | -76% (less market exposure) |

### The Alpha Story

Part 1's negative alpha (-0.062) with high beta (1.146) meant the strategy was just a leveraged market position — a "beta masquerade." Any returns came from riding the market, not from skill.

Part 2 flips this: **positive alpha (0.093) with low beta (0.278)**. The CNN is generating independent returns. The low beta comes from the 55% confidence threshold — when uncertain, the model sits out, reducing systematic exposure.

In [ ]:
# RUNNABLE: Visualize Part 1 vs Part 2 performance
import numpy as np

# Verified backtest results
metrics = {
    'Part 1 (Ridge)': {
        'Net Profit': 34.8,
        'Sharpe': 0.212,
        'Alpha': -0.062,
        'Beta': 1.146,
        'Max DD': 54.7,
    },
    'Part 2 (CNN)': {
        'Net Profit': 145.2,
        'Sharpe': 0.649,
        'Alpha': 0.093,
        'Beta': 0.278,
        'Max DD': 26.3,
    }
}

print("Performance Comparison: Classical ML vs Deep Learning")
print("="*60)
print(f"{'Metric':<15} {'Part 1 (Ridge)':>15} {'Part 2 (CNN)':>15} {'Δ':>12}")
print("-"*60)

for metric in ['Net Profit', 'Sharpe', 'Alpha', 'Beta', 'Max DD']:
    v1 = metrics['Part 1 (Ridge)'][metric]
    v2 = metrics['Part 2 (CNN)'][metric]
    
    if metric == 'Max DD':
        delta = f"{v2-v1:+.1f}%"  # Lower is better
        print(f"{metric:<15} {v1:>14.1f}% {v2:>14.1f}% {delta:>12}")
    elif metric == 'Net Profit':
        delta = f"{(v2/v1 - 1)*100:+.0f}%"
        print(f"{metric:<15} {v1:>14.1f}% {v2:>14.1f}% {delta:>12}")
    else:
        delta = f"{v2-v1:+.3f}"
        print(f"{metric:<15} {v1:>15.3f} {v2:>15.3f} {delta:>12}")

print("\n→ CNN generates genuine alpha (positive) vs Ridge's beta masquerade (negative)")

## 6. Analysis

### Why Does the CNN Work?

**1. Feature Learning vs. Feature Engineering**

Part 1 used hand-picked features (volatility, ATR, open interest). If those features don't contain signal, no model can find it. The CNN learns features directly from raw OHLCV — it can discover patterns we didn't think to look for.

**2. Nonlinear Relationships**

Ridge regression can only learn linear relationships: "if volatility is high, weight is low." The CNN can learn complex patterns: "if yesterday was a gap-up reversal with high volume followed by a narrow-range day, predict DOWN."

**3. Temporal Structure**

The temporal split lets the model treat recent and distant patterns differently. Market memory decays — what happened 2 days ago matters more than what happened 14 days ago. The CNN can learn this decay implicitly.

### Why Might It *Not* Generalize?

**1. Narrow Universe**

Training on only 3 mega-cap tech stocks means the model learns patterns specific to those stocks. Apple's patterns may not transfer to small-cap biotech.

**2. Regime Dependence**

The backtest covers 2019–2024: a historic bull market, COVID crash, recovery, inflation scare, AI boom. These regimes may not repeat. The model learned patterns that worked *in this period* — will they work in the next?

**3. Statistical Significance**

The Probabilistic Sharpe Ratio (PSR) is 21.9% — well below the 95% threshold. We can't confidently say the Sharpe is significantly different from zero. With ~5 years of weekly data (~260 samples), we need a Sharpe above ~0.8 for statistical significance.

> **Key Insight:** A positive backtest is necessary but not sufficient. The PSR tells us we need more data (or a higher Sharpe) to be confident this isn't luck.

## Try This: Exercises

Deepen your understanding by modifying the strategy. All exercises work on QC's free tier.

### Exercise 1: Change the Confidence Threshold

The algorithm requires >55% confidence to trade. What if you lower it?

```python
# In _trade(), change:
confidence > 0.55
# To:
confidence > 0.40  # Trade on weaker signals
```

**Predict:** Will Sharpe improve or decline? Will beta increase?

---

### Exercise 2: Expand the Universe

The strategy trades only 3 stocks. What if you add more?

```python
# In initialize(), change:
self._universe_size = self.get_parameter("universe_size", 3)
# To:
self._universe_size = self.get_parameter("universe_size", 10)
```

**Predict:** More diversification, but more models to train. Will the alpha hold up across a broader universe?

---

### Exercise 3: Remove Temporal Splitting

The temporal split is the key architectural innovation. Remove it to see if it matters:

```python
# In _create_model(), replace the temporal split with:
flattened = Flatten()(feature_extraction)  # Skip the split entirely
outputs = Dense(3, activation='softmax')(flattened)
```

**Predict:** If the temporal split captures real market structure, removing it should hurt performance.

## 7. References

### Code

- [Full algorithm source (GitHub)](https://github.com/ababber/pyhou-02-17-2026/tree/main/part-2-deep-learning)
- [QuantConnect documentation](https://www.quantconnect.com/docs)

### Books

- Chollet, F. (2021). *Deep Learning with Python, 2nd ed.* — Chapters 8 (CNNs), 4 (getting started), 2 (tensor math)
- Géron, A. (2022). *Hands-On Machine Learning, 3rd ed.* — Chapter 14 (CNNs), Chapter 10 (ANNs)
- Jansen, S. (2020). *Machine Learning for Algorithmic Trading, 2nd ed.* — Chapter 18 (CNNs for time series)
- Orr, M. (2023). *Math for Programmers* — Chapter 16 (neural networks from scratch)
- Peixeiro, M. (2022). *Time Series Forecasting in Python* — Chapter 3 (deep learning for TSF)

### Papers

- LeCun, Y. et al. (1989). Backpropagation applied to handwritten zip code recognition. *Neural Computation*.
- Bailey, D. & López de Prado, M. (2012). The Sharpe ratio efficient frontier. *Journal of Risk*. (PSR)

---

**Next:** Part 3 explores foundation models — pre-trained transformers that predict time series without any task-specific training. Can zero-shot forecasting beat our trained CNN?